# Analysis of pp events: event activity observables

Firstly, we load libraries.

In [8]:
import ROOT
import numpy as np

We also enable multi-threading, which allows event processing across multiple CPU cores, and connect RDataFrame to the TTree in our created events.root file.

In [9]:
ROOT.EnableImplicitMT() # enable multi-threading

df = ROOT.RDataFrame("events", "events.root") # connect RDataFrame to the TTree in events.root

### Multiplicity distribution

In [3]:
df_with_Nch = df.Define("Nch", "pT.size()")

histo_Nch = df_with_Nch.Histo1D(("histo_Nch", "Multiplicity;N_{ch};Normalized Events", 70, 0, 70), "Nch") # histogram of the multiplicity distribution
histo_normalized_Nch = histo_Nch.GetValue() # get the histogram object from the RDataFrame
canvas_mult =  ROOT.TCanvas("canvas_mult", "Multiplicity", 800, 600) # create a canvas to draw the histogram
canvas_mult.SetLogy() # set logarithmic scale for y-axis

if histo_Nch.Integral() > 0: # check if the histogram has entries to avoid division by zero
    histo_normalized_Nch.Scale(1.0 / histo_Nch.Integral()) # normalize the histogram
histo_normalized_Nch.SetLineColor(ROOT.kRed + 1) # set line color to slightly darker (+1) red
histo_normalized_Nch.SetFillColorAlpha(ROOT.kRed, 0.1) # set fill color to red with 10% transparency (alpha = 0.1)

histo_normalized_Nch.Draw("HIST") # draw the histogram
canvas_mult.SaveAs("img/multiplicity.pdf") # save the canvas as a PDF file

Info in <TCanvas::Print>: pdf file img/multiplicity.pdf has been created


### Unweighted Spherocity $S_0^{p_\mathrm{T} = 1}$ distribution

calculated only for $N_\mathrm{ch}(|\eta| < 1) > 10$

$$
S_0^{p_\mathrm{T} = 1} = \frac{\pi^2}{4} \min_{\hat{n}} \left( \frac{\sum_i |\hat{u}_{\mathrm{T}, i} \times \hat{n}|}{N_\mathrm{trks}} \right)^2
$$

In [4]:
# define px, py using pT, phi; multiplicity as the number of pT values in each event; ux, uy using px, py, pT
df_with_px_py_Nch_ux_uy = (
    df.Define("px", "pT * cos(phi)")
      .Define("py", "pT * sin(phi)")
      .Define("Nch", "pT.size()")
      .Define("ux", "px / pT")
      .Define("uy", "py / pT")
)

In [5]:
# USING ONLY PYTHON AND AVOIDING C++ (sufficient for small data sets)

# extract px, py, pT. and multiplicity from the RDataFrame into a dictionary of NumPy arrays -- this loads the data into RAM!!! BE CAREFUL WITH LARGE DATASETS --> better option is to write a function in C++ inside PyROOT as Gemini showed me, but since I'm learning, I will do it this way for now
data = df_with_px_py_Nch_ux_uy.AsNumpy(["ux", "uy", "Nch"])

# define each NumPy arrays separately
uxArr = data["ux"]
uyArr = data["uy"]
NchArr = data["Nch"]

# define the function to calculate the unweighted transverse spherocity
def calc_S0pT1_slow(uxEvent, uyEvent, NchEvent):

    # Since the minimizing axis \hat{n} is always collinear with one of the pT vectors in the event, we can calculate the sum for each pT vector and find the minimum. This is much faster than performing a numerical minimization. 
    absCrossProductMatrix = np.abs(uxEvent[:, np.newaxis]*uyEvent - uyEvent[:, np.newaxis]*uxEvent) # calculate the absolute value of the cross product of each pair of unit pT vectors utilizing [:, np.newaxis], which turns a 1D array into a column vector
    sums = np.sum(absCrossProductMatrix, axis=0) # calculate the sums for each axis (pT vector)
    minSum = np.min(sums) # find the minimum sum

    return np.pi**2 / 4 * (minSum / NchEvent)**2 # calculate and return the unweighted transverse spherocity

# prepare ROOT histogram
canvas_S0pT1_slow = ROOT.TCanvas("canvas_S0pT1_slow", "S0pT1_slow", 800, 600) # create a canvas to draw the histogram
histo_S0pT1_slow = ROOT.TH1F("histo_S0pT1_slow", "Unweighted Transverse Spherocity;S_{0}^{p_{T}=1};Normalized Events", 100, 0, 1) # create a histogram for S0pT1 distribution

# event loop
for iEvent in range(len(NchArr)):
    # define observable arrays for each event
    uxEvent = uxArr[iEvent]
    uyEvent = uyArr[iEvent]
    NchEvent = NchArr[iEvent]

    if not NchEvent > 10: # select events with multiplicity > 10
        continue
    else:
        S0pT1_slow = calc_S0pT1_slow(uxEvent, uyEvent, NchEvent) # calculate the unweighted transverse spherocity for the event
        histo_S0pT1_slow.Fill(S0pT1_slow) # fill the histogram

# customize the histogram and save it
canvas_S0pT1_slow.SetLogy() # set logarithmic scale for y-axis

if histo_S0pT1_slow.Integral() > 0: # check if the histogram is not empty to avoid division by zero
    histo_S0pT1_slow.Scale(1.0 / histo_S0pT1_slow.Integral()) # normalize the histogram
histo_S0pT1_slow.SetLineColor(ROOT.kRed + 1) # set line color to slightly darker (+1) red
histo_S0pT1_slow.SetFillColorAlpha(ROOT.kRed, 0.1) # set fill color to red with 10% transparency (alpha = 0.1)

histo_S0pT1_slow.Draw("HIST") # draw the histogram
canvas_S0pT1_slow.SaveAs("img/S0pT1_slow.pdf") # save the canvas as a PDF file

Info in <TCanvas::Print>: pdf file img/S0pT1_slow.pdf has been created


In [7]:
# USING C++ IN PyROOT

# I wrote the calc_S0pT1(px, py, pT, Nch) function in C++, now we need to tell ROOT to load it and compile it. 
ROOT.gInterpreter.ProcessLine('#include "cpp/S0pT1.cpp"')

df_S0pT1 = (df_with_px_py_Nch_ux_uy.Filter("Nch > 10", "multiplicity cut")
                                   .Define("S0pT1", "calc_S0pT1(ux, uy, Nch)")
)

histo_S0pT1 = df_S0pT1.Histo1D(("histo_S0pT1", "Unweighted Transverse Spherocity;S_{0}^{p_{T} = 1};Normalized Events", 100, 0, 1), "S0pT1") # histogram of the unweighted spherocity distribution
histo_normalized_S0pT1 = histo_S0pT1.GetValue() # get the histogram object from the RDataFrame
canvas_S0pT1 =  ROOT.TCanvas("canvas_S0pT1", "S0pT1", 800, 600) # create a canvas to draw the histogram
canvas_S0pT1.SetLogy() # set logarithmic scale for y-axis

if histo_S0pT1.Integral() > 0: # check if the histogram has entries to avoid division by zero
    histo_normalized_S0pT1.Scale(1.0 / histo_S0pT1.Integral()) # normalize the histogram
histo_normalized_S0pT1.SetLineColor(ROOT.kRed + 1) # set line color to slightly darker (+1) red
histo_normalized_S0pT1.SetFillColorAlpha(ROOT.kRed, 0.1) # set fill color to red with 10% transparency (alpha = 0.1)

histo_normalized_S0pT1.Draw("HIST") # draw the histogram
canvas_S0pT1.SaveAs("img/S0pT1.pdf") # save the canvas as a PDF file

In file included from input_line_170:1:
./cpp/S0pT1.cpp:8:8: error: redefinition of 'calc_S0pT1'
double calc_S0pT1(const ROOT::RVec<float>& uxEvent, // const is a safety lock to ensure that the function only reads data and does not change it
       ^
./cpp/S0pT1.cpp:8:8: note: previous definition is here
double calc_S0pT1(const ROOT::RVec<float>& uxEvent, // const is a safety lock to ensure that the function only reads data and does not change it
       ^
Info in <TCanvas::Print>: pdf file img/S0pT1.pdf has been created
